In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import json
from typing import Union


def read_json(in_file: Union[Path, str]):
    if not isinstance(in_file, Path):
        in_file = Path(in_file)
    return json.loads(in_file.read_text(encoding='utf8'))

In [ ]:
METRIC_LIST = ['depth_ratio', 'ops_ratio']#, 'cx_ratio']

def wide_to_long(df):
    df = pd.melt(
        df,
        id_vars=[col for col in df.columns if col not in METRIC_LIST],
        value_vars=METRIC_LIST,
        var_name='metric',
        value_name='value'
    )
    return df

In [4]:
def get_ours_metrics(dir:  Path):
    if not isinstance(dir, Path):
        dir = Path(dir)
    assert dir.is_dir(), dir

    records = []
    for subdir in dir.iterdir():
        try:
            metrics = read_json(subdir / 'metrics.json')
            config = read_json(subdir / 'config.json')
        except:
            continue
        rec = {key.replace('metric/', ''): value for key, value in metrics.items()}
        rec.update(graph_name=config['hardware'].lower(),
                   circuit=config['circuit'],
                   layout_method=config['init_strategy'].lower() + '-ours',
                   opt_method='ours',
                   routing_method='ours',
                   opt_order='ours')
        records.append(rec)

    return pd.DataFrame.from_records(records)

df_ours = wide_to_long(get_ours_metrics('./output/ours/nam_circs'))

In [6]:
df_ours.circuit.unique()

array(['barenco_tof_10', 'barenco_tof_3', 'csla_mux_3', 'csum_mux_9',
       'gf2^4_mult', 'gf2^5_mult', 'gf2^6_mult', 'ham15-low', 'hwb6',
       'mod_red_21', 'qcla_com_7', 'rc_adder_6', 'tof_10', 'tof_3',
       'tof_4', 'tof_5', 'vbe_adder_3', 'barenco_tof_4', 'barenco_tof_5'],
      dtype=object)

In [7]:
df_ours.graph_name.unique()

array(['star', 'line', 'grid', 'ring'], dtype=object)

In [15]:
df_baseline = wide_to_long(
    pd.concat([
        pd.read_csv('./output/baseline/qiskit_name_circs.csv'),
        ]))
df_baseline['circuit'] = df_baseline['circuit'].str.replace('.qasm', '')

In [17]:
df_baseline.circuit.unique()

array(['barenco_tof_10', 'barenco_tof_3', 'csla_mux_3', 'csum_mux_9',
       'gf2^4_mult', 'gf2^5_mult', 'gf2^6_mult', 'ham15-low', 'hwb6',
       'mod5_4', 'mod_mult_55', 'mod_red_21', 'qcla_com_7', 'rc_adder_6',
       'tof_10', 'tof_3', 'tof_4', 'tof_5', 'vbe_adder_3',
       'barenco_tof_4', 'barenco_tof_5'], dtype=object)

In [16]:
df_baseline.opt_order.unique()

array(['both', 'before', 'after'], dtype=object)

In [8]:
def plot_bar_comparison(df, title, x, hue=None):
    plt.figure(figsize=(15,3))
    plt.suptitle(title)

    for i, metric in enumerate(METRIC_LIST):
        ax = plt.subplot(1,len(METRIC_LIST),i+1)
        sns.barplot(df[df.metric == metric], y='value', x=x, ax=ax, legend=bool(hue), hue=hue)
        ax.set_ylabel('')
        ax.set_xlabel(metric)
        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', label_type='edge', padding=3)